### Desafios SQL misturando:
- JOINs
- CTEs
- Window Functions
- agregações
- filtros reais de negócio

In [42]:
import pandas as pd
import sqlite3

In [43]:
connection = sqlite3.connect("./db/ecommerce2.db")
crsr = connection.cursor()
print("Connected to the database")

Connected to the database


In [44]:
clientes_data = {
  'id': [1, 2, 3, 4],
  'nome': ['Ana', 'Bruno', 'Carla', 'Diego'],
  'cidade': ['São Paulo', 'Rio de Janeiro', 'Salvador', 'São Paulo']
}

pedidos_data = {
  'id': [1,2,3,4,5,6,7],
  'cliente_id': [1, 1, 2, 2, 3, 1, 4],
  'data_pedido': ['2024-01-10', '2024-02-05', '2024-01-15', '2024-03-01', '2024-02-20', '2024-03-10', '2024-03-15'],
  'valor': [300.0, 150.0, 500.0, 200.0, 700.0, 100.0, 400.0]
}

pagamentos_data = {
  'id': [1, 2, 3, 4, 5, 6, 7],
  'pedido_id': [1, 2, 3, 4, 5, 6, 7],
  'status': ['pago', 'pago', 'pendente', 'pago', 'pago', 'cancelado', 'pago']
}

In [45]:
df_clientes = pd.DataFrame(clientes_data)
df_pedidos = pd.DataFrame(pedidos_data)
df_pagamentos = pd.DataFrame(pagamentos_data)

In [46]:
df_clientes.to_sql('clientes', connection, if_exists='replace', index=False)
df_pedidos.to_sql('pedidos', connection, if_exists='replace', index=False)
df_pagamentos.to_sql('pagamentos', connection, if_exists='replace', index=False)
print("DataFrames inserted into the database")

DataFrames inserted into the database


#### Exercicios 1

In [47]:
#Liste o nome de cada cliente e o valor total já pago em pedidos.
sql_query = '''
SELECT c.nome, SUM(p.valor) as total_pago
FROM pedidos as p
JOIN clientes as c ON p.cliente_id = c.id
JOIN pagamentos as pg ON p.id = pg.pedido_id
WHERE pg.status = 'pago'
GROUP BY c.nome
ORDER BY total_pago DESC
'''
df = pd.read_sql_query(sql_query, connection)
df.head(5)

,nome,total_pago
0,Carla,700.0
1,Ana,450.0
2,Diego,400.0
3,Bruno,200.0


In [48]:
'''Crie uma CTE com os pedidos pagos e depois mostre:

cidade
quantidade de pedidos pagos
valor total pago por cidade'''
sql_query = '''
WITH pedidos_pagos AS (
SELECT pedido_id
FROM pagamentos
WHERE status = 'pago'
)
SELECT c.cidade,
COUNT(*) AS pedidos_pagos,
SUM(p.valor) AS total_pago
FROM pedidos_pagos pg
JOIN pedidos p ON pg.pedido_id = p.id
JOIN clientes c ON p.cliente_id = c.id
GROUP BY c.cidade
ORDER BY total_pago DESC;
'''
df = pd.read_sql_query(sql_query, connection)
df.head(5)

,cidade,pedidos_pagos,total_pago
0,São Paulo,3,850.0
1,Salvador,1,700.0
2,Rio de Janeiro,1,200.0


In [49]:
'''Para cada pedido, mostrar:

nome cliente
data_pedido
valor
ranking do maior pedido por cliente'''

sql_query = '''
SELECT c.nome, p.data_pedido, p.valor, RANK() OVER (PARTITION BY c.nome ORDER BY p.valor DESC) as rank
FROM pedidos as p
JOIN clientes as c ON p.cliente_id = c.id
'''
df = pd.read_sql_query(sql_query, connection)
df.head(5)

,nome,data_pedido,valor,rank
0,Ana,2024-01-10,300.0,1
1,Ana,2024-02-05,150.0,2
2,Ana,2024-03-10,100.0,3
3,Bruno,2024-01-15,500.0,1
4,Bruno,2024-03-01,200.0,2


In [50]:
#Descubra o maior pedido pago de cada cidade.
sql_query = '''
WITH pagos AS (
SELECT c.cidade, c.nome, p.valor, ROW_NUMBER() OVER ( PARTITION BY c.cidade ORDER BY p.valor DESC) AS rn
FROM pedidos p
JOIN clientes c ON p.cliente_id = c.id
JOIN pagamentos pg ON pg.pedido_id = p.id
WHERE pg.status = 'pago'
)
SELECT cidade, nome, valor
FROM pagos
WHERE rn = 1;
'''
df = pd.read_sql_query(sql_query, connection)
df.head(5)

,cidade,nome,valor
0,Rio de Janeiro,Bruno,200.0
1,Salvador,Carla,700.0
2,São Paulo,Diego,400.0


In [51]:
#Mostrar histórico de compras da cliente Ana
sql_query = '''
SELECT P.data_pedido, p.valor, SUM(p.valor) OVER (ORDER BY p.data_pedido ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as total_acumulado
FROM pedidos as p
JOIN clientes as c ON p.cliente_id = c.id
WHERE c.id = 1;
'''
df = pd.read_sql_query(sql_query, connection)
df.head(5)

,data_pedido,valor,total_acumulado
0,2024-01-10,300.0,300.0
1,2024-02-05,150.0,450.0
2,2024-03-10,100.0,550.0


In [52]:
#Quais clientes gastaram acima da média geral dos clientes considerando apenas pedidos pagos?
sql_query = '''
WITH total_clientes AS (
SELECT c.nome, SUM(p.valor) AS total_pago
FROM pagamentos pg
JOIN pedidos p ON pg.pedido_id = p.id
JOIN clientes c ON p.cliente_id = c.id
WHERE pg.status = 'pago'
GROUP BY c.nome
),
media_geral AS (
SELECT AVG(total_pago) AS media_clientes
FROM total_clientes )
SELECT tc.nome,tc.total_pago
FROM total_clientes tc
CROSS JOIN media_geral mg
WHERE tc.total_pago > mg.media_clientes
ORDER BY tc.total_pago DESC;
'''
df = pd.read_sql_query(sql_query, connection)
df.head(5)

,nome,total_pago
0,Carla,700.0
1,Ana,450.0


#### Exercícios 2

In [53]:
vendas_data = {
  'id': [1, 2, 3, 4, 5, 6, 7, 8, 9],
  'vendedor': ['Ana', 'Ana', 'Ana', 'Bruno', 'Bruno', 'Carla', 'Carla', 'Diego', 'Diego'],
  'regiao': ['Sul', 'Sul', 'Sul', 'Norte', 'Norte', 'Sul', 'Sul', 'Norte', 'Norte'],
  'data_venda': ['2024-01-05', '2024-01-20', '2024-02-10', '2024-01-08', '2024-02-01', '2024-01-15', '2024-02-18', '2024-02-25', '2024-03-02'],
  'valor': [500.0, 300.0, 700.0, 400.0, 600.0, 900.0, 200.0, 1000.0, 300.0]
}
df_vendas = pd.DataFrame(vendas_data)
df_vendas.to_sql('vendas', connection, if_exists='replace', index=False)

9

In [54]:
#Ranking por vendedor
sql_query = '''
SELECT vendedor, data_venda, valor, RANK() OVER (PARTITION BY vendedor ORDER BY valor DESC) as rank
FROM vendas
'''
df = pd.read_sql_query(sql_query, connection)
df.head(10)

,vendedor,data_venda,valor,rank
0,Ana,2024-02-10,700.0,1
1,Ana,2024-01-05,500.0,2
2,Ana,2024-01-20,300.0,3
3,Bruno,2024-02-01,600.0,1
4,Bruno,2024-01-08,400.0,2
5,Carla,2024-01-15,900.0,1
6,Carla,2024-02-18,200.0,2
7,Diego,2024-02-25,1000.0,1
8,Diego,2024-03-02,300.0,2


In [55]:
#Maior venda por região
sql_query = '''
WITH rank_vendas AS (
SELECT regiao, vendedor, valor, ROW_NUMBER() OVER (PARTITION BY regiao ORDER BY valor DESC) as rn
FROM vendas)
SELECT * FROM rank_vendas
WHERE rn = 1;
'''
df = pd.read_sql_query(sql_query, connection)
df.head(5)

,regiao,vendedor,valor,rn
0,Norte,Diego,1000.0,1
1,Sul,Carla,900.0,1


In [56]:
#Acumulado por vendedor
sql_query = '''
SELECT vendedor, data_venda, valor, SUM(valor) OVER (PARTITION BY vendedor ORDER BY data_venda) as total_acumulado
FROM vendas
'''
df = pd.read_sql_query(sql_query, connection)
df.head(10)

,vendedor,data_venda,valor,total_acumulado
0,Ana,2024-01-05,500.0,500.0
1,Ana,2024-01-20,300.0,800.0
2,Ana,2024-02-10,700.0,1500.0
3,Bruno,2024-01-08,400.0,400.0
4,Bruno,2024-02-01,600.0,1000.0
5,Carla,2024-01-15,900.0,900.0
6,Carla,2024-02-18,200.0,1100.0
7,Diego,2024-02-25,1000.0,1000.0
8,Diego,2024-03-02,300.0,1300.0


In [57]:
#Comparação com venda anterior
sql_query = '''
SELECT vendedor, data_venda, valor, LAG(valor) OVER (PARTITION BY vendedor ORDER BY data_venda) as valor_anterior, valor - LAG(valor) OVER (PARTITION BY vendedor ORDER BY data_venda) as diferenca
FROM vendas
'''
df = pd.read_sql_query(sql_query, connection)
df.head(10)

,vendedor,data_venda,valor,valor_anterior,diferenca
0,Ana,2024-01-05,500.0,NaN,NaN
1,Ana,2024-01-20,300.0,500.0,-200.0
2,Ana,2024-02-10,700.0,300.0,400.0
3,Bruno,2024-01-08,400.0,NaN,NaN
4,Bruno,2024-02-01,600.0,400.0,200.0
5,Carla,2024-01-15,900.0,NaN,NaN
6,Carla,2024-02-18,200.0,900.0,-700.0
7,Diego,2024-02-25,1000.0,NaN,NaN
8,Diego,2024-03-02,300.0,1000.0,-700.0


In [58]:
#Média e acima da média
sql_query = '''
WITH base AS (
SELECT vendedor,data_venda,valor,AVG(valor) OVER (PARTITION BY vendedor) AS media_vendedor
FROM vendas)
SELECT *
FROM base
WHERE valor > media_vendedor;
'''
df = pd.read_sql_query(sql_query, connection)
df.head(10)

,vendedor,data_venda,valor,media_vendedor
0,Ana,2024-02-10,700.0,500.0
1,Bruno,2024-02-01,600.0,500.0
2,Carla,2024-01-15,900.0,550.0
3,Diego,2024-02-25,1000.0,650.0


In [59]:
#Qual vendedor teve maior crescimento entre sua primeira e última venda?
sql_query = '''
WITH vendas_rank AS (
SELECT vendedor,data_venda, valor, ROW_NUMBER() OVER (PARTITION BY vendedor ORDER BY data_venda) AS rn_inicio,
ROW_NUMBER() OVER (PARTITION BY vendedor ORDER BY data_venda DESC) AS rn_fim
FROM vendas
),
primeira AS (
SELECT vendedor, valor AS primeira_venda
FROM vendas_rank
WHERE rn_inicio = 1
),
ultima AS (
SELECT vendedor, valor AS ultima_venda
FROM vendas_rank
WHERE rn_fim = 1
)
SELECT p.vendedor,
p.primeira_venda,
u.ultima_venda,
u.ultima_venda - p.primeira_venda AS crescimento
FROM primeira p
JOIN ultima u
ON p.vendedor = u.vendedor
ORDER BY crescimento DESC;
'''
df = pd.read_sql_query(sql_query, connection)
df.head(5)

,vendedor,primeira_venda,ultima_venda,crescimento
0,Ana,500.0,700.0,200.0
1,Bruno,400.0,600.0,200.0
2,Carla,900.0,200.0,-700.0
3,Diego,1000.0,300.0,-700.0


In [60]:
connection.close()
print("Connection closed")

Connection closed
